In [65]:
import pandas as pd

In [66]:
data = pd.read_json('exercises.json')
data.head()
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 876 entries, 0 to 875
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   name              876 non-null    str   
 1   force             846 non-null    str   
 2   level             876 non-null    str   
 3   mechanic          789 non-null    str   
 4   equipment         799 non-null    str   
 5   primaryMuscles    876 non-null    object
 6   secondaryMuscles  876 non-null    object
 7   instructions      876 non-null    object
 8   category          876 non-null    str   
 9   images            876 non-null    object
 10  id                876 non-null    str   
dtypes: object(4), str(7)
memory usage: 75.4+ KB


In [67]:
data['primaryMuscles'] = data['primaryMuscles'].str[0]
data['instructions'] = data['instructions'].str.join('\n')
data = data.drop(labels=['images','id'],axis=1)

In [68]:
data.head(1)

,name,force,level,mechanic,equipment,primaryMuscles,secondaryMuscles,instructions,category
0,3/4 Sit-Up,pull,beginner,compound,body only,abdominals,[],Lie down on the floor and secure your feet. Yo...,strength


In [69]:
data = data.rename(columns={'force':'movement_type','primaryMuscles':'primary_muscle','category':'exercise_category'})
data = data.drop(labels=['mechanic','secondaryMuscles'],axis=1)

In [71]:
import os, sys
os.chdir('..')            # move to backend/ so config.py can find .env
sys.path.append('.')      # so `import app` works

from sqlalchemy import insert
from app.database import SessionLocal
from app.models import Exercise

# pandas uses NaN for missing values; the DB needs None, which becomes NULL
records = data.astype(object).where(data.notna(), None).to_dict('records')

with SessionLocal() as db:
    db.execute(insert(Exercise), records)
    db.commit()

print(f'inserted {len(records)} exercises')


IntegrityError: (psycopg.errors.UniqueViolation) duplicate key value violates unique constraint "exercises_name_key"
DETAIL:  Key (name)=(3/4 Sit-Up) already exists.
[SQL: INSERT INTO exercises (id, name, instructions, equipment, level, movement_type, primary_muscle, exercise_category) VALUES (%(id)s::UUID, %(name)s::VARCHAR, %(instructions)s::VARCHAR, %(equipment)s::VARCHAR, %(level)s::VARCHAR, %(movement_type)s::VARCHAR, %(primary_muscle)s::VARCHAR, %(exercise_category)s::VARCHAR)]
[parameters: [{'id': UUID('d0c2df79-6c84-48a5-a2a3-88f58a135795'), 'name': '3/4 Sit-Up', 'instructions': 'Lie down on the floor and secure your feet. Your legs should be bent at the knees.\nPlace your hands behind or to the side of your head. You will beg ... (167 characters truncated) ... our torso should be perpendicular to the ground. Reverse the motion, going only ¾ of the way down.\nRepeat for the recommended amount of repetitions.', 'equipment': 'body only', 'level': 'beginner', 'movement_type': 'pul